In [2]:
%cd /mlx_devbox/users/janne.spijkervet/repo/samantha
%load_ext autoreload
%autoreload 2

/mlx_devbox/users/janne.spijkervet/repo/samantha


In [3]:
import torch
import torchaudio
from recipes.umm.modules.semantic_module import TextSemanticStage
from recipes.soundstorm2.lightning.soundstorm import SoundStorm, SoundStormConfig
from recipes.soundstorm2.lightning.soundstream import SoundStreamSpeech24k
from recipes.soundstorm2.lightning.bestrq import BestRQMelCTCModel

In [5]:
device = "cuda"


In [ ]:
from recipes.datasets.libritts import LibriTTSWebDataModule

pl_datamodule = LibriTTSWebDataModule(
    sample_rate=24000,
    batch_size=8,
    shuffle_buffer_size=100
)

train_loader = pl_datamodule.train_dataloader()

In [4]:
batch = next(iter(train_loader))

23/08/09 18:11:52 WARN hdfs.DFSClient: Detect slow read. Current read node is FDBD:DC02:28:40A::32, which throughput is 1547942 B/s.Current read file is /home/byte_speech_sv/data/speech/libritts/train-clean-360/00001.tar. Current pos is 4587520. Current Block Id is 120439990006
23/08/09 18:12:00 WARN hdfs.DFSClient: Detect slow read. Current read node is FDBD:DC02:22:297::24, which throughput is 3141715 B/s.Current read file is /home/byte_speech_sv/data/speech/libritts/train-clean-360/00000.tar. Current pos is 33292288. Current Block Id is 120439999115
23/08/09 18:12:23 WARN hdfs.DFSClient: Detect slow read. Current read node is FDBD:DC02:28:40A::32, which throughput is 3143021 B/s.Current read file is /home/byte_speech_sv/data/speech/libritts/train-clean-360/00001.tar. Current pos is 109576192. Current Block Id is 120439990006
23/08/09 18:12:44 WARN hdfs.DFSClient: Detect slow read. Current read node is FDBD:DC02:26:160::13, which throughput is 3143076 B/s.Current read file is /home/b

In [5]:
batch = {k: v.to(device) if type(v) == torch.Tensor else v for k, v in batch.items()}

cat: Unable to write to output stream.


# SoundStorm Stage

In [4]:
audio_model = SoundStreamSpeech24k()
semantic_model = BestRQMelCTCModel()

2023-08-09 18:41:11,992 - recipes.soundstorm2.lightning.soundstream - WARNING - Loading checkpoint from HDFS: hdfs:///home/byte_speech_sv/models/soundstream/soundstream_speech_24k/soundstream_speech_24k_encoder.pt...
2023-08-09 18:41:20,314 - recipes.soundstorm2.lightning.soundstream - WARNING - Loading checkpoint from HDFS: hdfs:///home/byte_speech_sv/models/soundstream/soundstream_speech_24k/soundstream_speech_24k_decoder.pt...
2023-08-09 18:41:28,491 - recipes.soundstorm2.lightning.bestrq - WARNING - Loading checkpoint from HDFS: step=020000-tr_loss=0.3844-val_loss_0=1.3190.ckpt...


NameError: name 'device' is not defined

In [6]:
soundstorm_ckpt_path = "last.ckpt"
soundstorm = SoundStorm.load_from_checkpoint(soundstorm_ckpt_path, audio_model=audio_model, semantic_model=semantic_model).to(device)

In [7]:
iterations = [48, 32, 24, 16, 8, 4, 2, 2, 1, 1, 1, 1]
score_strategies = ["random", "random", "random", "random", "maskgit", "maskgit","maskgit","maskgit","maskgit","maskgit","maskgit","maskgit",]
guidance_scale = None
temperatures = [1.0, 1.0, 0.95, 0.95, 0.9, 0.9, 0.8, 0.8, 0.4, 0.4, 0.4, 0.4]

In [10]:
inputs = soundstorm.prepare_inputs(batch)

In [11]:
inputs["semantic_tokens"]

tensor([[29674, 23841, 23841,  ...,   126,   126,   126],
        [10921,  4479, 20042,  ..., 20042, 20042, 20042],
        [ 3564, 13248, 32330,  ...,  1371, 32454,  2631],
        ...,
        [15235, 24895, 29705,  ..., 32380, 32380, 15124],
        [29986,  6202,  6202,  ...,  6202,  6202,  6202],
        [30050,  3927,  3927,  ...,  3927,  3927,  3927]], device='cuda:0')

In [12]:
mels = semantic_model.model.model.model_input_transform(batch["audio"], normalize=None)

In [13]:
mels.shape

torch.Size([8, 1387, 80])

In [37]:
import matplotlib.pyplot as plt
from samantha.transforms.audio import batch_plot_spectrogram
from IPython import display as ipd

idx = 1
semantic_tokens = inputs["semantic_tokens"][idx:idx+1]
text = inputs["normalized_text"][idx:idx+1]
audio = inputs["audio"][idx:idx+1]
print(text)
# fig, ax = plt.subplots(8, 1, figsize=(20,10))
# batch_plot_spectrogram(mels[idx:idx+1].permute(0, 2, 1), plot_log=False, mel=True, ax=ax)

# semantic_tokens = semantic_tokens[:, :250]
max_seq_len = semantic_tokens.shape[1] * 2
audio_tokens, _ = soundstorm.iterative_decoding(
    max_seq_len=max_seq_len,
    iterations=iterations,
    score_strategies=score_strategies,
    semantic_tokens=semantic_tokens,
    guidance_scale=guidance_scale,
    temperatures=temperatures,
    sampled_t=None
)
with torch.no_grad():
    sampled_audio = soundstorm.audio_model.decode(audio_tokens)

ipd.display(ipd.Audio(audio[0].cpu(), rate=24000))
ipd.display(ipd.Audio(sampled_audio[0].cpu(), rate=24000))

['It happened all at once, retreat and continuation for a moment somehow combined. And, if he did not definitely see the awful thing, at least he was aware that it had come to pass.']


Iteratively decoding audio tokens...: 100%|█████| 12/12 [00:04<00:00,  2.44it/s]


# Semantic Stage

In [8]:
from recipes.umm.modules.semantic_module import TextSemanticStage

In [26]:
# semantic_stage_ckpt_path = "step=030000-loss=0.0000-val_loss_0=0.0000.ckpt"
semantic_stage_ckpt_path = "step=010000-loss=0.0000-val_loss_0=0.0000.ckpt"
semantic_stage = TextSemanticStage.load_from_checkpoint(semantic_stage_ckpt_path).to(device)

2023-08-09 19:12:28,183 - recipes.soundstorm2.lightning.bestrq - WARNING - Loading checkpoint from HDFS: step=020000-tr_loss=0.3844-val_loss_0=1.3190.ckpt...


In [32]:
semantic_temperature = 1.0

In [41]:
text = ["It happened all at once, retreat and continuation for a moment somehow combined, where the door closes, another one opens"]
semantic_tokens = semantic_stage.generate(text, max_seq_len=250, temperature=semantic_temperature)



Sampling semantic tokens...:  56%|██████▋     | 139/250 [00:02<00:01, 60.82it/s]


In [42]:
print(semantic_tokens.min(), semantic_tokens.max())
print(semantic_tokens.max() < soundstorm.semantic_model.codebook_size)

tensor(495, device='cuda:0') tensor(32746, device='cuda:0')
tensor(True, device='cuda:0')


In [43]:
max_seq_len = semantic_tokens.shape[1] * 2
print(max_seq_len)
audio_tokens, _ = soundstorm.iterative_decoding(
    max_seq_len=max_seq_len,
    iterations=iterations,
    score_strategies=score_strategies,
    semantic_tokens=semantic_tokens,
    guidance_scale=guidance_scale,
    temperatures=temperatures,
    sampled_t=None
)

278


Iteratively decoding audio tokens...: 100%|█████| 12/12 [00:02<00:00,  5.04it/s]


In [44]:
import IPython.display as ipd
with torch.no_grad():
    sampled_audio = soundstorm.audio_model.decode(audio_tokens)
ipd.display(ipd.Audio(sampled_audio[0].cpu(), rate=24000))